 # LSTM — PyTorch Implementation Notes
#
 1. LSTM INPUT
 - Input shape with batch_first=True:
       [batch_size, sequence_length, input_size]
#
 - Example:
       x.shape = [4, 8, 10]
  → 4 sequences in a batch
   → each sequence has 8 time steps
   → each time step has 10 features
#
#
 2. LSTM PARAMETERS
 - input_size:
       Number of features at each time step.
#
 - hidden_size:
       Number of hidden features in the hidden state.
       This determines the size of the representation produced by the LSTM.
#
 - num_layers:
       Number of stacked LSTM layers.
#
 - batch_first=True:
       Makes the input/output format:
       [batch_size, sequence_length, features]
#
#
 3. LSTM OUTPUTS
 - nn.LSTM returns three things:
#
       lstm_out, (h_n, c_n)
#
 - lstm_out:
       Hidden output at EVERY time step from the LAST LSTM layer.
#
       Shape:
       [batch_size, sequence_length, hidden_size]
#
 - h_n:
       Final hidden state from EVERY LSTM layer.
#
       Shape:
       [num_layers, batch_size, hidden_size]
#
 - c_n:
       Final cell state from EVERY LSTM layer.
#
       Shape:
       [num_layers, batch_size, hidden_size]
#
#
 4. HIDDEN STATE vs CELL STATE
 - h_t = hidden state
       Short-term/current representation and output of the LSTM.
#
 - c_t = cell state
       Long-term memory carried through the sequence.
#
 - At every time step, the LSTM uses gates to decide what information
   to forget, what new information to store, and what information to
   expose through the hidden state.
#
 - PyTorch's nn.LSTM performs these gate calculations internally.
#
#
 5. USING THE FINAL OUTPUT FOR PREDICTION
 - lstm_out contains outputs for all time steps.
#
 - To use the final time step:
#
       lstm_out[:, -1, :]
#
 - This gives the final hidden representation from the LAST LSTM layer.
#
 - It can then be passed through a Linear layer:
#
       final hidden representation → Linear → prediction
#
 - Alternatively, h_n[-1] gives the final hidden state of the last layer.
#
#
 6. TRAINING LOOP
 - Each epoch performs:
#
       Forward pass
           ↓
       Calculate loss
           ↓
       Zero old gradients
           ↓
       Backpropagation
           ↓
       Update parameters
#
 - The forward pass must happen INSIDE the epoch loop so that every
   epoch creates a fresh computation graph using the updated parameters.
#
#
 7. LOSS
 - MSELoss was used because the example produces continuous outputs.
#
 - The loss decreased during training, showing that the optimizer
   was updating the model parameters.
#
#
 8. KEY TAKEAWAY
 - nn.LSTM handles the internal LSTM mechanism:
       input → gates → cell state → hidden state
#
 - The implementation focuses on understanding:
       architecture
       input/output shapes
       hidden state
       cell state
       final timestep output
       training with backpropagation

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim

In [32]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size=input_size,
                          hidden_size=hidden_size,
                          num_layers=num_layers,
                          batch_first=True)

        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        lstm_out, (h_n, c_n) = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        output = self.fc(last_output)
        return output

In [33]:
model = LSTMModel(input_size=10, hidden_size=20, output_size=5, num_layers=2)
x = torch.randn(4, 8, 10)
y = torch.randn(4,5)


In [34]:
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)
epochs = 10
for epoch in range(epochs):
    predictions = model(x)
    loss = criterion(predictions, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [1/10], Loss: 1.2984
Epoch [2/10], Loss: 1.2539
Epoch [3/10], Loss: 1.2145
Epoch [4/10], Loss: 1.1795
Epoch [5/10], Loss: 1.1483
Epoch [6/10], Loss: 1.1205
Epoch [7/10], Loss: 1.0956
Epoch [8/10], Loss: 1.0734
Epoch [9/10], Loss: 1.0536
Epoch [10/10], Loss: 1.0358
